<a href="https://colab.research.google.com/github/Dulandi-Dinethya/SE4050-Deep_Learning_Assignment/blob/IT23185302/results/models/02_VGG16_IT23185302.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VGG16-Based Palmprint Recognition

**Module:** SE4050 – Deep Learning  
**Student ID:** IT23185302  
**Model:** VGG16  

## Project
**Deep Learning-Based Palmprint Recognition System for Contactless Biometric Identification and Authentication of Registered Users**

## Objective
This notebook implements and evaluates a VGG16-based deep learning model for palmprint recognition. Transfer learning with a VGG16 model pretrained on ImageNet is used to extract visual features from palm images and classify them according to registered user identities.

## 1. Environment Setup and Library Imports

The required Python libraries for data handling, visualization, deep learning, and model evaluation are imported in this section. TensorFlow/Keras is used to implement the VGG16 model, while scikit-learn is used to calculate classification performance metrics.

In [1]:
# Basic libraries
import os
import random
import time

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# VGG16 model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


### 1.1 Reproducibility

A fixed random seed is used to improve the reproducibility of the experiment.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


## 2. Dataset and Prepared Data Loading

The palm images were preprocessed using the common preprocessing
pipeline developed for the project. The successfully processed images
from Session 1 and Session 2 are loaded for the VGG16 experiment.
Samples that failed ROI extraction during preprocessing are excluded
from model training.

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_DRIVE = "/content/drive/MyDrive/SE4050_Palm_Dataset"

SESSION1_ZIP = os.path.join(DATA_DRIVE, "session1.zip")
SESSION2_ZIP = os.path.join(DATA_DRIVE, "session2.zip")

print("Session 1 ZIP exists:", os.path.exists(SESSION1_ZIP))
print("Session 2 ZIP exists:", os.path.exists(SESSION2_ZIP))

Session 1 ZIP exists: True
Session 2 ZIP exists: True


In [5]:
import zipfile

EXTRACT_DIR = "/content/palm_preprocessed"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(SESSION1_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

with zipfile.ZipFile(SESSION2_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extraction completed.")

Dataset extraction completed.


In [6]:
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, "").count(os.sep)
    indent = "    " * level

    print(
        f"{indent}{os.path.basename(root)}/ "
        f"({len(files)} files, {len(dirs)} folders)"
    )

    if level >= 2:
        dirs[:] = []

palm_preprocessed/ (0 files, 2 folders)
    session1/ (5994 files, 0 folders)
    session2/ (5994 files, 0 folders)


In [7]:
image_extensions = (".png", ".jpg", ".jpeg", ".tiff", ".tif")

all_images = []

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith(image_extensions):
            all_images.append(os.path.join(root, file))

print("Total usable image files found:", len(all_images))
print("\nFirst 10 files:")

for path in all_images[:10]:
    print(path)

Total usable image files found: 11988

First 10 files:
/content/palm_preprocessed/session1/05157.png
/content/palm_preprocessed/session1/00085.png
/content/palm_preprocessed/session1/04931.png
/content/palm_preprocessed/session1/03848.png
/content/palm_preprocessed/session1/05429.png
/content/palm_preprocessed/session1/04035.png
/content/palm_preprocessed/session1/01482.png
/content/palm_preprocessed/session1/02441.png
/content/palm_preprocessed/session1/04835.png
/content/palm_preprocessed/session1/04520.png


## 3. Dataset Structure and Identity Verification

The extracted preprocessed dataset contains palm images from two acquisition
sessions. Before creating the training, validation, and test sets, the dataset
structure and filename patterns are examined to determine the correct identity
labels and avoid data leakage between subsets.

In [8]:
SESSION1_DIR = os.path.join(EXTRACT_DIR, "session1")
SESSION2_DIR = os.path.join(EXTRACT_DIR, "session2")

session1_files = sorted([
    f for f in os.listdir(SESSION1_DIR)
    if f.lower().endswith(".png")
])

session2_files = sorted([
    f for f in os.listdir(SESSION2_DIR)
    if f.lower().endswith(".png")
])

print("Session 1 images:", len(session1_files))
print("Session 2 images:", len(session2_files))

print("\nFirst 20 Session 1 filenames:")
print(session1_files[:20])

print("\nFirst 20 Session 2 filenames:")
print(session2_files[:20])

Session 1 images: 5994
Session 2 images: 5994

First 20 Session 1 filenames:
['00001.png', '00002.png', '00003.png', '00004.png', '00005.png', '00006.png', '00007.png', '00008.png', '00009.png', '00010.png', '00011.png', '00012.png', '00013.png', '00014.png', '00015.png', '00016.png', '00017.png', '00018.png', '00019.png', '00020.png']

First 20 Session 2 filenames:
['00001.png', '00002.png', '00003.png', '00004.png', '00005.png', '00006.png', '00007.png', '00008.png', '00009.png', '00010.png', '00011.png', '00012.png', '00013.png', '00014.png', '00015.png', '00016.png', '00017.png', '00018.png', '00019.png', '00020.png']


In [9]:
print("Last 20 Session 1 filenames:")
print(session1_files[-20:])

print("\nLast 20 Session 2 filenames:")
print(session2_files[-20:])

Last 20 Session 1 filenames:
['05981.png', '05982.png', '05983.png', '05984.png', '05985.png', '05986.png', '05987.png', '05988.png', '05989.png', '05990.png', '05991.png', '05992.png', '05993.png', '05994.png', '05995.png', '05996.png', '05997.png', '05998.png', '05999.png', '06000.png']

Last 20 Session 2 filenames:
['05981.png', '05982.png', '05983.png', '05984.png', '05985.png', '05986.png', '05987.png', '05988.png', '05989.png', '05990.png', '05991.png', '05992.png', '05993.png', '05994.png', '05995.png', '05996.png', '05997.png', '05998.png', '05999.png', '06000.png']


In [10]:
print("Filename length examples:")

for f in session1_files[:20]:
    print(
        f,
        "| stem:", os.path.splitext(f)[0],
        "| length:", len(os.path.splitext(f)[0])
    )

Filename length examples:
00001.png | stem: 00001 | length: 5
00002.png | stem: 00002 | length: 5
00003.png | stem: 00003 | length: 5
00004.png | stem: 00004 | length: 5
00005.png | stem: 00005 | length: 5
00006.png | stem: 00006 | length: 5
00007.png | stem: 00007 | length: 5
00008.png | stem: 00008 | length: 5
00009.png | stem: 00009 | length: 5
00010.png | stem: 00010 | length: 5
00011.png | stem: 00011 | length: 5
00012.png | stem: 00012 | length: 5
00013.png | stem: 00013 | length: 5
00014.png | stem: 00014 | length: 5
00015.png | stem: 00015 | length: 5
00016.png | stem: 00016 | length: 5
00017.png | stem: 00017 | length: 5
00018.png | stem: 00018 | length: 5
00019.png | stem: 00019 | length: 5
00020.png | stem: 00020 | length: 5


In [11]:
metadata_files = []

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith((".csv", ".txt", ".json")):
            metadata_files.append(os.path.join(root, file))

print("Metadata files found:", len(metadata_files))

for path in metadata_files:
    print(path)

Metadata files found: 0


In [12]:
expected = {f"{i:05d}.png" for i in range(1, 6001)}

s1_set = set(session1_files)
s2_set = set(session2_files)

missing_s1 = sorted(expected - s1_set)
missing_s2 = sorted(expected - s2_set)

print("Missing from Session 1:", missing_s1)
print("Missing from Session 2:", missing_s2)

print("\nMissing count Session 1:", len(missing_s1))
print("Missing count Session 2:", len(missing_s2))

Missing from Session 1: ['02795.png', '02798.png', '02897.png', '05444.png', '05452.png', '05457.png']
Missing from Session 2: ['02432.png', '02655.png', '02656.png', '04506.png', '04507.png', '05215.png']

Missing count Session 1: 6
Missing count Session 2: 6


## 4. Identity Label Generation

The dataset contains 600 palm identities. The original filename numbering is
preserved after preprocessing so that removal of failed ROI samples does not
change the identity assignments. Identity labels are generated from the
original image numbers rather than from the position of an image in the
filtered dataset.

In [13]:
def get_identity(filename):
    """
    Convert the original image number to an identity label.

    Images 00001-00010 -> class 0
    Images 00011-00020 -> class 1
    ...
    Images 05991-06000 -> class 599
    """
    image_number = int(os.path.splitext(filename)[0])
    identity = (image_number - 1) // 10

    return identity

In [14]:
test_files = [
    "00001.png",
    "00010.png",
    "00011.png",
    "00020.png",
    "05991.png",
    "06000.png"
]

for file in test_files:
    print(file, "-> Class", get_identity(file))

00001.png -> Class 0
00010.png -> Class 0
00011.png -> Class 1
00020.png -> Class 1
05991.png -> Class 599
06000.png -> Class 599


In [15]:
records = []

for filename in session1_files:
    records.append({
        "filepath": os.path.join(SESSION1_DIR, filename),
        "filename": filename,
        "session": "session1",
        "label": get_identity(filename)
    })

for filename in session2_files:
    records.append({
        "filepath": os.path.join(SESSION2_DIR, filename),
        "filename": filename,
        "session": "session2",
        "label": get_identity(filename)
    })

df = pd.DataFrame(records)

print("Total samples:", len(df))
print("Number of classes:", df["label"].nunique())
print("Label range:", df["label"].min(), "-", df["label"].max())

df.head(10)

Total samples: 11988
Number of classes: 600
Label range: 0 - 599


,filepath,filename,session,label
0,/content/palm_preprocessed/session1/00001.png,00001.png,session1,0
1,/content/palm_preprocessed/session1/00002.png,00002.png,session1,0
2,/content/palm_preprocessed/session1/00003.png,00003.png,session1,0
3,/content/palm_preprocessed/session1/00004.png,00004.png,session1,0
4,/content/palm_preprocessed/session1/00005.png,00005.png,session1,0
5,/content/palm_preprocessed/session1/00006.png,00006.png,session1,0
6,/content/palm_preprocessed/session1/00007.png,00007.png,session1,0
7,/content/palm_preprocessed/session1/00008.png,00008.png,session1,0
8,/content/palm_preprocessed/session1/00009.png,00009.png,session1,0
9,/content/palm_preprocessed/session1/00010.png,00010.png,session1,0


In [16]:
print("Session 1 samples:", len(df[df["session"] == "session1"]))
print("Session 2 samples:", len(df[df["session"] == "session2"]))

print("\nSession 1 classes:",
      df[df["session"] == "session1"]["label"].nunique())

print("Session 2 classes:",
      df[df["session"] == "session2"]["label"].nunique())

Session 1 samples: 5994
Session 2 samples: 5994

Session 1 classes: 600
Session 2 classes: 600


In [17]:
from sklearn.model_selection import train_test_split

# Separate the two sessions
session1_df = df[df["session"] == "session1"].copy()
session2_df = df[df["session"] == "session2"].copy()

# Split Session 1 into 80% training and 20% validation
train_df, val_df = train_test_split(
    session1_df,
    test_size=0.20,
    random_state=SEED,
    stratify=session1_df["label"]
)

# Keep Session 2 completely separate for final testing
test_df = session2_df.copy()

# Display split information
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining classes:", train_df["label"].nunique())
print("Validation classes:", val_df["label"].nunique())
print("Testing classes:", test_df["label"].nunique())

Training samples: 4795
Validation samples: 1199
Testing samples: 5994

Training classes: 600
Validation classes: 600
Testing classes: 600


In [18]:
train_paths = set(train_df["filepath"])
val_paths = set(val_df["filepath"])
test_paths = set(test_df["filepath"])

print("Train ∩ Validation:", len(train_paths & val_paths))
print("Train ∩ Test:", len(train_paths & test_paths))
print("Validation ∩ Test:", len(val_paths & test_paths))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


## 5. TensorFlow Data Pipeline

The prepared training, validation, and test splits are converted into
TensorFlow datasets. Each palm image is decoded, converted to RGB,
resized to 224 × 224 pixels, and paired with its corresponding
identity label.

In [19]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    # Read image
    image = tf.io.read_file(filepath)

    # Decode PNG as RGB
    image = tf.image.decode_png(image, channels=3)

    # Resize for VGG16
    image = tf.image.resize(image, IMG_SIZE)

    # Convert to float32
    image = tf.cast(image, tf.float32)

    return image, label


def create_dataset(dataframe, training=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    # Shuffle only training data
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("TensorFlow datasets created successfully.")

TensorFlow datasets created successfully.


In [20]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image data type:", images.dtype)
    print("Label range in batch:",
          int(tf.reduce_min(labels)),
          "-",
          int(tf.reduce_max(labels)))

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Image data type: <dtype: 'float32'>
Label range in batch: 3 - 538


## 6. Data Augmentation

Data augmentation is applied only to the training images to improve
generalization and reduce overfitting. Small transformations are used
while preserving the important palm-line characteristics required for
identity recognition.

In [21]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    )
], name="data_augmentation")

print("Data augmentation pipeline created successfully.")

Data augmentation pipeline created successfully.


## 7. VGG16 Input Preprocessing

The input images are prepared using the preprocessing function provided
for the pretrained VGG16 network. Data augmentation is applied only to
the training set, while validation and test images are kept unchanged
apart from VGG16-specific preprocessing.

In [22]:
def prepare_train(images, labels):
    # Apply augmentation only to training images
    images = data_augmentation(images, training=True)

    # Apply VGG16-specific preprocessing
    images = preprocess_input(images)

    return images, labels


def prepare_eval(images, labels):
    # No augmentation for validation/test data
    images = preprocess_input(images)

    return images, labels


# Apply preprocessing
train_vgg_ds = train_ds.map(
    prepare_train,
    num_parallel_calls=AUTOTUNE
)

val_vgg_ds = val_ds.map(
    prepare_eval,
    num_parallel_calls=AUTOTUNE
)

test_vgg_ds = test_ds.map(
    prepare_eval,
    num_parallel_calls=AUTOTUNE
)

# Improve pipeline performance
train_vgg_ds = train_vgg_ds.prefetch(AUTOTUNE)
val_vgg_ds = val_vgg_ds.prefetch(AUTOTUNE)
test_vgg_ds = test_vgg_ds.prefetch(AUTOTUNE)

print("VGG16 preprocessing applied successfully.")

VGG16 preprocessing applied successfully.


## 8. Pretrained VGG16 Base Model

A VGG16 network pretrained on the ImageNet dataset is used as the
feature extractor. The original VGG16 classification layers are excluded
so that a new classification head can be constructed for the 600 palm
identity classes.

In [23]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

print("Pretrained VGG16 base model loaded successfully.")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Pretrained VGG16 base model loaded successfully.


## 9. Freeze the VGG16 Base Model

The pretrained VGG16 convolutional base is frozen during the initial
training stage. This preserves the pretrained ImageNet features while
the newly added classification layers learn to distinguish the 600
palm identities.

In [24]:
# Freeze all pretrained VGG16 layers
base_model.trainable = False

print("VGG16 base model trainable:", base_model.trainable)
print("Total VGG16 layers:", len(base_model.layers))
print("Trainable weights:", len(base_model.trainable_weights))

VGG16 base model trainable: False
Total VGG16 layers: 19
Trainable weights: 0


## 10. VGG16 Classification Head

A custom classification head is added to the frozen VGG16 feature
extractor. Global average pooling reduces the extracted feature maps,
followed by a 512-unit dense layer with ReLU activation and dropout
for regularization. The final softmax layer contains 600 units,
corresponding to the 600 palm identity classes.

In [25]:
NUM_CLASSES = 600

inputs = keras.Input(shape=(224, 224, 3))

# Frozen VGG16 feature extractor
x = base_model(inputs, training=False)

# Custom classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)

# 600 palm identity classes
outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="VGG16_Palm_Recognition"
)

print("VGG16 classification model created successfully.")

VGG16 classification model created successfully.


In [26]:
model.summary()

Model: "VGG16_Palm_Recognition"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 600)            │       307,800 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,285,144 (58.31 MB)

 Trainable params: 570,456 (2.18 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

## 11. Model Compilation

The VGG16 transfer-learning model is compiled using the Adam optimizer
with a learning rate of 0.0001. Sparse categorical cross-entropy is used
because the palm identity labels are represented as integer class indices
from 0 to 599. Accuracy is used as the primary training metric.

In [27]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("VGG16 model compiled successfully.")

VGG16 model compiled successfully.


## 12. Training Callbacks

Training callbacks are used to improve training reliability. ModelCheckpoint
saves the best-performing model based on validation accuracy, EarlyStopping
stops training when validation performance no longer improves, and
ReduceLROnPlateau reduces the learning rate when validation loss stops
improving.

In [28]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

CHECKPOINT_PATH = "/content/best_vgg16_palm.keras"

checkpoint = ModelCheckpoint(
    CHECKPOINT_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

callbacks = [
    checkpoint,
    early_stopping,
    reduce_lr
]

print("Training callbacks created successfully.")

Training callbacks created successfully.


In [29]:
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 13. Initial Training

The model is initially trained with the pretrained VGG16 convolutional
base frozen. During this stage, only the newly added classification
head is updated. Validation performance is monitored using the configured
callbacks.

In [30]:
INITIAL_EPOCHS = 20

history_initial = model.fit(
    train_vgg_ds,
    validation_data=val_vgg_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 576ms/step - accuracy: 6.0985e-04 - loss: 7.3477
Epoch 1: val_accuracy improved from None to 0.00667, saving model to /content/best_vgg16_palm.keras

Epoch 1: finished saving model to /content/best_vgg16_palm.keras
150/150 ━━━━━━━━━━━━━━━━━━━━ 134s 713ms/step - accuracy: 8.3420e-04 - loss: 7.0019 - val_accuracy: 0.0067 - val_loss: 6.3607 - learning_rate: 1.0000e-04
Epoch 2/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 474ms/step - accuracy: 0.0047 - loss: 6.3849
Epoch 2: val_accuracy improved from 0.00667 to 0.01418, saving model to /content/best_vgg16_palm.keras

Epoch 2: finished saving model to /content/best_vgg16_palm.keras
150/150 ━━━━━━━━━━━━━━━━━━━━ 81s 536ms/step - accuracy: 0.0052 - loss: 6.3069 - val_accuracy: 0.0142 - val_loss: 6.0925 - learning_rate: 1.0000e-04
Epoch 3/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 464ms/step - accuracy: 0.0139 - loss: 6.0603
Epoch 3: val_accuracy improved from 0.01418 to 0.05421, saving model to /content/best_vgg16_palm

## 14. VGG16 Fine-Tuning

After the initial transfer-learning stage, the final convolutional block
of VGG16 is unfrozen for fine-tuning. Earlier VGG16 layers remain frozen
to preserve general pretrained visual features. A smaller learning rate
is used during fine-tuning to gradually adapt the higher-level VGG16
features to palmprint recognition.

In [31]:
# Enable the VGG16 base model for fine-tuning
base_model.trainable = True

# Freeze all VGG16 layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze only VGG16 Block 5
for layer in base_model.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True

print("Total VGG16 layers:", len(base_model.layers))
print("Trainable VGG16 layers:")

for layer in base_model.layers:
    if layer.trainable:
        print(" -", layer.name)

Total VGG16 layers: 19
Trainable VGG16 layers:
 - block5_conv1
 - block5_conv2
 - block5_conv3
 - block5_pool


### Fine-Tuning Compilation

The model is recompiled using a lower learning rate of 0.00001 before
fine-tuning. This allows the unfrozen VGG16 layers to adapt gradually
to palmprint-specific features without making large changes to the
pretrained weights.

In [32]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model recompiled for fine-tuning.")
print("Fine-tuning learning rate: 1e-5")

Model recompiled for fine-tuning.
Fine-tuning learning rate: 1e-5


### Fine-Tuning Callbacks

Separate callbacks are configured for the fine-tuning stage. The best
fine-tuned model is saved according to validation accuracy, while early
stopping and learning-rate reduction help prevent unnecessary training
and overfitting.

In [33]:
FINE_TUNE_CHECKPOINT = "/content/best_vgg16_finetuned.keras"

fine_checkpoint = ModelCheckpoint(
    FINE_TUNE_CHECKPOINT,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

fine_early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

fine_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

fine_callbacks = [
    fine_checkpoint,
    fine_early_stopping,
    fine_reduce_lr
]

print("Fine-tuning callbacks configured successfully.")

Fine-tuning callbacks configured successfully.


## 15. Fine-Tuning Training

The model is fine-tuned by training the final VGG16 convolutional block
together with the custom classification head. A lower learning rate is
used to gradually adapt the pretrained high-level features to palmprint
identity recognition.

In [34]:
FINE_TUNE_EPOCHS = 10

history_fine = model.fit(
    train_vgg_ds,
    validation_data=val_vgg_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=fine_callbacks
)

Epoch 1/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 502ms/step - accuracy: 0.5258 - loss: 2.1115
Epoch 1: val_accuracy improved from None to 0.71476, saving model to /content/best_vgg16_finetuned.keras

Epoch 1: finished saving model to /content/best_vgg16_finetuned.keras
150/150 ━━━━━━━━━━━━━━━━━━━━ 96s 577ms/step - accuracy: 0.5418 - loss: 2.0347 - val_accuracy: 0.7148 - val_loss: 1.6211 - learning_rate: 1.0000e-05
Epoch 2/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 480ms/step - accuracy: 0.5865 - loss: 1.7858
Epoch 2: val_accuracy did not improve from 0.71476
150/150 ━━━━━━━━━━━━━━━━━━━━ 83s 550ms/step - accuracy: 0.5962 - loss: 1.7506 - val_accuracy: 0.7056 - val_loss: 1.5983 - learning_rate: 1.0000e-05
Epoch 3/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 462ms/step - accuracy: 0.6168 - loss: 1.6582
Epoch 3: val_accuracy improved from 0.71476 to 0.75480, saving model to /content/best_vgg16_finetuned.keras

Epoch 3: finished saving model to /content/best_vgg16_finetuned.keras
150/150 ━━━━━━━━━━━━━━━━━━━━ 80s 5